# Discover a schema from your documents

You have a pile of rental listings and want them as a table, but no list of columns yet. Let TypeLLM write the schema: show it a few sample listings and ask for the fields, each with a name, a type and a description. Then turn those fields into questions and extract every listing with them.

1. Describe a field: the meta schema.
2. Discover the fields from three samples.
3. Turn the fields into questions.
4. Extract every listing.

Paste your TypeLLM API key in the cell below, then run all cells.

In [1]:
%pip install -q "typellm>=0.6.9" pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Paste your API key here.
TYPELLM_API_KEY = ""  # https://typellm.ai/dashboard/keys

In [3]:
from typellm import TypeLLMClient

client = TypeLLMClient(api_key=TYPELLM_API_KEY)

LISTINGS = [
    "Sunny 2BR/1BA in Mission District, $3,450/mo. 850 sq ft, hardwood floors, in-unit washer/dryer. Cats OK, no dogs. Available Nov 1. 12-month lease, deposit one month's rent. Street parking only.",
    "Studio near Lake Merritt, Oakland. $1,795 a month, utilities included except electricity. 420 sq ft, 3rd floor walk-up. No pets. Move in Oct 20. Laundry in basement.",
    "Spacious 3 bed 2 bath house in Berkeley hills - $5,200/month. Garage for 2 cars, private yard, dogs welcome (max 2). 1,600 sq ft. Lease 12 months min, $7,800 deposit. Available December 1st.",
    "Furnished 1 bedroom in SoMa high-rise, 24th floor with bay views. $3,900/mo, 6-month lease OK. Gym, rooftop deck, doorman. One parking spot included. Small pets under 25 lb allowed. Available now.",
    "Room in shared 4BR flat, Inner Sunset. $1,350/mo + share of utilities (~$80). Shared kitchen and bath, washer in unit. No pets, no smoking. Month-to-month. Available Nov 15.",
    "2 bed 2 bath condo in Palo Alto, $4,600 per month. 1,050 sq ft, renovated kitchen, in-unit laundry, EV charger in assigned spot. No pets. 1-year lease. Available January 5.",
]

## Describe a field: the meta schema

The meta schema is an array of fields, up to 30. Each field has a name, a type and a description.

In [4]:
META_SCHEMA = {
    "type": "array",
    "maxItems": 30,
    "items": {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "type": {"type": "string", "enum": ["string", "integer", "number", "boolean"]},
            "description": {"type": "string"},
        },
    },
    "instructions": "The fields of a table that holds one row per document like these. "
                    "One field per fact the documents state, named in snake_case.",
}

## Discover the fields from three samples

Pick samples that differ, so the fields cover what the others say too.

In [5]:
samples = "\n\n".join(LISTINGS[i] for i in (0, 1, 3))
fields = client.generate(context=f"Sample documents:\n\n{samples}", questions={"fields": META_SCHEMA}).result["fields"]

for field in fields:
    print(f"{field['name']:20s} {field['type']:8s} {field['description']}")

price_per_month      number   The monthly rental price in USD.
bedrooms             integer  The number of bedrooms in the unit.
bathrooms            integer  The number of bathrooms in the unit.
square_feet          integer  The total area of the unit in square feet.
location             string   The neighborhood or specific area where the unit is located.
available_date       string   The date when the unit becomes available for move-in.
pets_allowed         boolean  Whether pets are permitted in the unit.
furnished            boolean  Whether the unit is furnished.
lease_term_months    integer  The minimum lease duration in months.
parking_included     boolean  Whether a parking spot is included in the rental price.
deposit_months       number   The security deposit required, expressed as a multiple of the monthly rent.
floor_number         integer  The floor number of the unit within the building.
in_unit_laundry      boolean  Whether the unit has a washer and dryer inside the apar

## Turn the fields into questions

Each field becomes a question. Any listing may leave a field out, so every field takes null. Review the fields first if you like: rename, drop or add some.

In [6]:
QUESTIONS = {field["name"]: {"type": [field["type"], "null"], "instructions": field["description"]}
             for field in fields}

## Extract every listing

In [7]:
import pandas as pd

rows = [client.generate(context=listing, questions=QUESTIONS).result for listing in LISTINGS]
table = pd.DataFrame(rows).convert_dtypes()
print(table.iloc[:, :6].astype(object).fillna("").to_string(index=False))

price_per_month bedrooms bathrooms square_feet              location available_date
           3450        2         1         850      Mission District          Nov 1
           1795                            420 Lake Merritt, Oakland         Oct 20
           5200        3         2        1600        Berkeley hills   December 1st
           3900        1                                        SoMa            now
           1350        4                                Inner Sunset         Nov 15
           4600        2         2        1050             Palo Alto      January 5


In [8]:
table

,price_per_month,bedrooms,bathrooms,square_feet,location,available_date,pets_allowed,furnished,lease_term_months,parking_included,deposit_months,floor_number,in_unit_laundry,utilities_included,street_parking_only
0,3450,2,1,850,Mission District,Nov 1,True,<NA>,12,False,1.0,<NA>,True,<NA>,True
1,1795,<NA>,<NA>,420,"Lake Merritt, Oakland",Oct 20,False,<NA>,<NA>,<NA>,<NA>,3,False,False,<NA>
2,5200,3,2,1600,Berkeley hills,December 1st,True,<NA>,12,True,1.5,<NA>,<NA>,<NA>,False
3,3900,1,<NA>,<NA>,SoMa,now,True,True,6,True,<NA>,24,<NA>,<NA>,False
4,1350,4,<NA>,<NA>,Inner Sunset,Nov 15,False,<NA>,<NA>,False,<NA>,<NA>,False,False,<NA>
5,4600,2,2,1050,Palo Alto,January 5,False,<NA>,12,True,<NA>,<NA>,True,<NA>,False
